# Smoke test: training pipeline rehearsal (run before `sbatch`)

Rehearses the **exact** SLURM code path — `pipelines.training.run_full_cohort_event` and
`pipelines.training.run_feature_comp_task` — on a handful of representative events, before
submitting the ~1978-row manifest. This is not a reimplementation: it shells out to the same
entrypoints `array_full_cohort_run.sh` / `array_feature_comp.sh` call, with the same
`--max-iter 5000 --backend threading` the arrays use (argparse's own default is `1000` —
using it here would silently rehearse a different config than the cluster runs).

**Run this interactively on the cluster** (login/interactive node), not on a laptop — it needs
real `FEATURE_PATH` data and the same environment the array jobs run in.

**Why this exists.** A green SLURM run does not mean the fits were healthy. Today:
- No `logging.basicConfig` anywhere in `pipelines/training/` or `survival/`, so every
  per-alpha CV failure (`logger.debug`) and row-drop count (`logger.info`) is discarded at
  Python's default WARNING level.
- `ConvergenceWarning`/`RuntimeWarning` are globally suppressed
  (`survival/cox_models/_common.py`), so non-convergence is unmeasurable from stderr.
- A failed final fit writes an all-NaN `*_test.csv` and still **exits 0** — and
  `build_slurm_manifests --skip-completed` then treats that event as done because the file
  exists.
- `get_heldout_risk_scores_CoxPH` gates its failure message on `verbose`, which no caller
  sets, so a failed fold silently writes NaN risk scores for ~20% of patients.
- `np.nanmean` over folds can crown an alpha scored on a single surviving fold, while
  `error_rate` is computed per l1-path, not per alpha.

This notebook reads the diagnostic columns that already exist in the CV/test CSVs and the
skip-report JSONL to surface all of the above, and ends with a single PASS/FAIL verdict.

**Runtime caveat.** The grid is 25 alphas × 2 l1-ratios × 5 folds and is not CLI-configurable
(`DEFAULT_ALPHAS`/`DEFAULT_L1_RATIOS` are hardcoded in `slurm_array_utils.py`).
`parallel_axis="auto"` resolves to `"fold"` with only 2 l1-ratios, capping useful parallelism
at 5 cores regardless of how many you request. 5 events × (1 full-cohort + 6 modalities) is a
real workload on an interactive node — expect this to take a while; `text` and `prs` dominate
wall time, so set `MODALITIES = ["text"]` below for a faster subset pass first.

## 1. Isolate output writes from real results

`FEATURE_PATH` (inputs) and `SURV_PATH` (outputs) are both derived from the same `DATA_PATH` /
`CTEP_DATA_PATH` root (see `config.py`) — there is no separate output-only override. Pointing
`CTEP_DATA_PATH` at an empty scratch dir would isolate outputs but also break every input read.

Instead, build a scratch root that **symlinks the real read-only input subtrees** (feature
files, embedding-prediction parquets) and leaves `results/` as a real, empty directory the
smoke test writes into. Nothing under the real `results/` tree is touched, so
`--skip-completed` on the real array jobs is unaffected and results here can be deleted freely
afterward.

In [ ]:
def run_subprocess(cmd: list[str], label: str) -> dict:
    """Run one pipeline entry point and retain output for the verdict cells."""
    _initialize_skip_report_tracking()
    print(f"\n{'=' * 72}\n{label}\n$ {' '.join(cmd)}\n{'=' * 72}")
    started = time.perf_counter()
    completed = subprocess.run(
        cmd, cwd=V2_ROOT, env=os.environ.copy(), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
    )
    elapsed = time.perf_counter() - started
    if completed.stdout:
        print(completed.stdout, end="" if completed.stdout.endswith("\n") else "\n")
    print(f"[{label}] exit={completed.returncode}, wall={elapsed:.1f}s")
    return {"returncode": completed.returncode, "wall_s": elapsed, "stdout": completed.stdout or ""}


def build_full_cohort_cmd(scheme: str, event: str) -> list[str]:
    cmd = [sys.executable, "-m", "pipelines.training.run_full_cohort_event",
           "--scheme", scheme, "--event", event, "--anchor", ANCHOR,
           "--max-iter", str(MAX_ITER), "--backend", BACKEND]
    if OVERWRITE:
        cmd.append("--overwrite")
    if N_JOBS is not None:
        cmd += ["--n-jobs", str(N_JOBS)]
    return cmd


SKIP_REPORT_DIR = None
_skip_report_seen_before = {}


def _initialize_skip_report_tracking() -> None:
    global SKIP_REPORT_DIR
    if SKIP_REPORT_DIR is not None:
        return
    # The setup cell imported config once to discover the real data root. Evict that
    # cached module now that CTEP_DATA_PATH points at scratch, so later diagnostics
    # imports (notably schemes) resolve paths under the isolated smoke-test tree.
    sys.modules.pop("config", None)
    SKIP_REPORT_DIR = SMOKE_ROOT / "time-to-event_analysis" / "results" / "skipped_events"
    if SKIP_REPORT_DIR.exists():
        for fp in SKIP_REPORT_DIR.glob("*.jsonl"):
            with open(fp) as handle:
                _skip_report_seen_before[fp.name] = sum(1 for _ in handle)


In [ ]:
from __future__ import annotations

import json
import logging
import os
import subprocess
import sys
import time
from pathlib import Path

import polars as pl

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
# Per-alpha CV fit failures (grid_search.py) are logged at DEBUG, not INFO -- uncomment to see them:
# logging.getLogger("survival.cox_models.grid_search").setLevel(logging.DEBUG)


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import config as _real_config  # real DATA_PATH, before any env override

REAL_DATA_PATH = Path(_real_config.DATA_PATH)
print(f"v2 root:       {V2_ROOT}")
print(f"Real DATA_PATH: {REAL_DATA_PATH}")

In [ ]:
# Scratch root for this smoke test run. Change SMOKE_ROOT to taste (e.g. a scratch/tmp
# filesystem with enough space for a handful of grid-search outputs -- a few hundred MB).
SMOKE_ROOT = Path(os.environ.get("SMOKE_TEST_ROOT", "/tmp")) / "clinical_text_embedding_smoke_test"

if SMOKE_ROOT.exists():
    # Re-running: keep existing symlinks, just make sure results/ exists. Does NOT delete
    # prior smoke results, so --overwrite semantics below still apply within this scratch tree.
    print(f"Reusing existing scratch root: {SMOKE_ROOT}")
else:
    SMOKE_ROOT.mkdir(parents=True)
    print(f"Created scratch root: {SMOKE_ROOT}")

READ_ONLY_SUBDIRS = [
    "code_data",
    "clinical_and_genomic_features",
    "batched_datasets",
    "time-to-event_analysis",  # symlinked whole, then results/ is overridden below
]
for name in READ_ONLY_SUBDIRS:
    src = REAL_DATA_PATH / name
    dst = SMOKE_ROOT / name
    if not src.exists():
        print(f"  [skip] {src} does not exist on this host")
        continue
    if dst.is_symlink() or dst.exists():
        continue
    dst.symlink_to(src)
    print(f"  linked {dst} -> {src}")

# Override results/ specifically: real dir is symlinked above via time-to-event_analysis/,
# so remove that one link and replace the parent with a real (writable) directory containing
# a symlink to everything EXCEPT results/.
surv_link = SMOKE_ROOT / "time-to-event_analysis"
if surv_link.is_symlink():
    surv_link.unlink()
    surv_link.mkdir()
    real_surv = REAL_DATA_PATH / "time-to-event_analysis"
    for child in real_surv.iterdir():
        if child.name == "results":
            continue
        (surv_link / child.name).symlink_to(child)
    (surv_link / "results").mkdir(exist_ok=True)
    print(f"  results/ isolated under {surv_link / 'results'} (real results/ untouched)")

os.environ["CTEP_DATA_PATH"] = str(SMOKE_ROOT)

# config.py reads CTEP_DATA_PATH at import time. The `_real_config` import above (cell 2) is
# used only to read REAL_DATA_PATH for the symlinks above, before this override -- it is never
# consulted again, so its stale SURV_PATH/FEATURE_PATH are harmless. Every `from schemes import
# ...` / `from config import ...` below this cell happens fresh, after the override, and will
# resolve against SMOKE_ROOT correctly. If you add a direct import of config/schemes/pipelines.*
# ABOVE this cell (before the override), restart the kernel -- that module's cached path
# constants would be stale for the rest of the session.
print(f"CTEP_DATA_PATH set to: {os.environ['CTEP_DATA_PATH']}")

## 2. Pick representative events

One event per requested category, taken from the real manifests so the set stays valid if
events change. Edit `EVENTS` directly, or use `pick_first_per_scheme()` below.

In [ ]:
EVENTS = [
    # (scheme, event, category)
    ("death_met", "death", "OS"),
    ("death_met", "boneM", "time-to-met"),
    ("phecode_post", "789", "phecode"),
    ("icd3_post", "R11", "ICD level-3"),
    ("icd4_post", "R11.0", "ICD level-4"),
]

ANCHOR = "treatment"
MODALITIES = ["stage", "treatment", "somatic", "prs", "text", "metburden"]  # or e.g. ["text"] for a fast pass
N_JOBS = None  # None => SLURM_CPUS_PER_TASK env var, falls back to 1
MAX_ITER = 5000  # matches array_*.sh COXNET_MAX_ITER default -- NOT argparse's 1000 default
BACKEND = "threading"
OVERWRITE = False


def pick_first_per_scheme(manifest_fp: str) -> list[tuple[str, str]]:
    """Read a real slurm_manifests TSV and take the first (scheme, event) row per scheme."""
    df = pl.read_csv(manifest_fp, separator="\t", has_header=False, new_columns=["scheme", "event"])
    return list(df.group_by("scheme", maintain_order=True).first().iter_rows())


MANIFEST_FP = V2_ROOT / "slurm" / "slurm_manifests" / "full_cohort_tasks.tsv"
if MANIFEST_FP.exists():
    manifest_df = pl.read_csv(MANIFEST_FP, separator="\t", has_header=False, new_columns=["scheme", "event"])
    missing = [(s, e) for s, e, _ in EVENTS if manifest_df.filter((pl.col("scheme") == s) & (pl.col("event") == e)).is_empty()]
    if missing:
        print(f"WARNING: not found in {MANIFEST_FP.name}: {missing}")
    else:
        print(f"All {len(EVENTS)} default events verified present in {MANIFEST_FP.name}")
else:
    print(f"WARNING: manifest not found at {MANIFEST_FP} -- skipping presence check")

for scheme, event, category in EVENTS:
    print(f"  {category:<14} {scheme}:{event}")

## 3. Run the real entrypoints as subprocesses

Same invocation the SLURM arrays use, one subprocess per (event, run-type) so a single
failure doesn't kill the loop. Captures stdout/stderr for the diagnostics pass below.

In [ ]:
def build_feature_comp_cmd(scheme: str, event: str, modality: str) -> list[str]:
    cmd = [sys.executable, "-m", "pipelines.training.run_feature_comp_task",
           "--scheme", scheme, "--event", event, "--modality", modality, "--anchor", ANCHOR,
           "--max-iter", str(MAX_ITER), "--backend", BACKEND]
    if OVERWRITE:
        cmd.append("--overwrite")
    if N_JOBS is not None:
        cmd += ["--n-jobs", str(N_JOBS)]
    return cmd


def build_full_cohort_risk_scores_cmd(scheme: str, event: str) -> list[str]:
    cmd = [sys.executable, "-m", "pipelines.training.run_full_cohort_risk_scores",
           "--scheme", scheme, "--event", event, "--anchor", ANCHOR,
           "--max-iter", str(MAX_ITER), "--backend", BACKEND]
    if OVERWRITE:
        cmd.append("--overwrite")
    if N_JOBS is not None:
        cmd += ["--n-jobs", str(N_JOBS)]
    return cmd


run_results: list[dict] = []
for scheme, event, category in EVENTS:
    r = run_subprocess(build_full_cohort_cmd(scheme, event), f"{category} full_cohort {scheme}:{event}")
    r.update(scheme=scheme, event=event, category=category, run_type="full_cohort", modality=None)
    run_results.append(r)

    r = run_subprocess(
        build_feature_comp_cmd(scheme, event, "all"),
        f"{category} feature_comps(all) {scheme}:{event}",
    )
    r.update(scheme=scheme, event=event, category=category, run_type="feature_comps", modality="all")
    run_results.append(r)

    # Depends on the full_cohort CV grid (text_val.csv) written above to pick hyperparameters,
    # so this must run after the full_cohort call for the same event.
    r = run_subprocess(
        build_full_cohort_risk_scores_cmd(scheme, event),
        f"{category} full_cohort_risk_scores {scheme}:{event}",
    )
    r.update(scheme=scheme, event=event, category=category, run_type="full_cohort_risk_scores", modality=None)
    run_results.append(r)

print(f"\nDone. {sum(r['returncode'] == 0 for r in run_results)}/{len(run_results)} subprocess calls exited 0.")

## 4. Diagnostics

The failure modes described at the top are silent at the process-exit level, so the report
below reads the same CSVs/JSONL the real pipeline already writes rather than trusting the
subprocess return code alone.

In [ ]:
from schemes import get_output_dir

VAL_DIAG_COLS = ["l1_ratio", "alpha", "mean_auc(t)", "error_rate", "fold_error_flags"]


def read_val_diagnostics(val_fp: Path) -> dict:
    if not val_fp.exists():
        return {"exists": False}
    df = pl.read_csv(val_fp)
    n_nan_auc = int(df["mean_auc(t)"].is_nan().sum() + df["mean_auc(t)"].is_null().sum())
    valid = df.filter(pl.col("mean_auc(t)").is_finite())
    out = {
        "exists": True,
        "n_grid_points": len(df),
        "n_nan_auc": n_nan_auc,
        "max_error_rate": float(df["error_rate"].max()) if "error_rate" in df.columns else None,
        "mean_error_rate": float(df["error_rate"].mean()) if "error_rate" in df.columns else None,
        "any_fold_error_flag": bool(df["fold_error_flags"].cast(pl.String).str.to_lowercase().str.contains("true").any()) if "fold_error_flags" in df.columns else None,
    }
    if not valid.is_empty():
        best = valid.sort("mean_auc(t)", descending=True).row(0, named=True)
        out["best_l1_ratio"] = float(best["l1_ratio"])
        out["best_alpha"] = float(best["alpha"])
        out["alpha_at_grid_boundary"] = bool(best["alpha"] <= 1.00001e-5 or best["alpha"] >= 0.99999)
    else:
        out["best_l1_ratio"] = out["best_alpha"] = out["alpha_at_grid_boundary"] = None
    return out


def read_test_diagnostics(test_fp: Path) -> dict:
    if not test_fp.exists():
        return {"exists": False}
    df = pl.read_csv(test_fp)
    metric_cols = [c for c in ["mean_auc(t)", "mean_ibs", "mean_c_index"] if c in df.columns]
    all_nan = bool(df.select(pl.all_horizontal([~pl.col(c).is_finite() for c in metric_cols])).to_series().all()) if metric_cols else None
    return {"exists": True, "all_metrics_nan": all_nan}


def read_risk_score_nan_frac(risk_fp: Path, score_col: str) -> float | None:
    if not risk_fp.exists():
        return None
    df = pl.read_csv(risk_fp)
    if score_col not in df.columns or df.is_empty():
        return None
    return float((~df[score_col].is_finite().fill_null(False)).mean())


def new_skip_report_rows() -> list[dict]:
    rows = []
    if not SKIP_REPORT_DIR.exists():
        return rows
    for fp in SKIP_REPORT_DIR.glob("*.jsonl"):
        seen = _skip_report_seen_before.get(fp.name, 0)
        with open(fp) as f:
            for i, line in enumerate(f):
                if i >= seen:
                    rows.append(json.loads(line))
    return rows


STDOUT_TAGS = ["[skip-data]", "Dropped", "constant feature columns", "Common feature cohort:", "[time]", "[error]"]


def tags_seen(stdout: str) -> list[str]:
    return [t for t in STDOUT_TAGS if t in stdout]

In [ ]:
summary_rows = []

for scheme, event, category in EVENTS:
    fc = next(r for r in run_results if r["scheme"] == scheme and r["event"] == event and r["run_type"] == "full_cohort")
    fc_risk = next(r for r in run_results if r["scheme"] == scheme and r["event"] == event and r["run_type"] == "full_cohort_risk_scores")
    fc_out = Path(get_output_dir(scheme, "full_cohort", ANCHOR)) / event
    fc_risk_out = Path(get_output_dir(scheme, "full_cohort_risk_scores", ANCHOR)) / event
    for model in ["text", "base"]:
        val = read_val_diagnostics(fc_out / f"{model}_val.csv")
        test = read_test_diagnostics(fc_out / f"{model}_test.csv")
        risk_nan = read_risk_score_nan_frac(fc_risk_out / f"{model}_risk_scores.csv", f"{model}_risk_score")
        summary_rows.append({
            "category": category, "scheme": scheme, "event": event, "run_type": "full_cohort",
            "model_or_modality": model,
            # fc covers CV/fit; fc_risk covers the separate held-out risk-score subprocess --
            # both must succeed for this row to be considered healthy.
            "subprocess_ok": fc["returncode"] == 0 and fc_risk["returncode"] == 0,
            "wall_s": round(fc["wall_s"] + fc_risk["wall_s"], 0), "val_exists": val.get("exists"),
            "n_nan_auc_of_grid": val.get("n_nan_auc"), "n_grid_points": val.get("n_grid_points"),
            "max_error_rate": val.get("max_error_rate"), "any_fold_error_flag": val.get("any_fold_error_flag"),
            "best_alpha": val.get("best_alpha"), "alpha_at_boundary": val.get("alpha_at_grid_boundary"),
            "test_all_nan": test.get("all_metrics_nan"), "risk_score_nan_frac": risk_nan,
            "stdout_tags": tags_seen(fc["stdout"]) + tags_seen(fc_risk["stdout"]),
        })

    fcomp = next(r for r in run_results if r["scheme"] == scheme and r["event"] == event and r["run_type"] == "feature_comps")
    fcomp_out = Path(get_output_dir(scheme, "feature_comps", ANCHOR)) / event
    fcomp_risk_dir = Path(get_output_dir(scheme, "feature_comps", ANCHOR)) / ".." / "held_out_risk_scores" / event
    fcomp_risk_dir = fcomp_risk_dir.resolve()
    for modality in MODALITIES:
        val = read_val_diagnostics(fcomp_out / f"{modality}_val.csv")
        test = read_test_diagnostics(fcomp_out / f"{modality}_test.csv")
        risk_nan = read_risk_score_nan_frac(fcomp_risk_dir / f"{modality}_risk_scores.csv", f"{modality}_risk_score")
        summary_rows.append({
            "category": category, "scheme": scheme, "event": event, "run_type": "feature_comps",
            "model_or_modality": modality,
            # the --modality all subprocess as a whole may report non-zero if ANY modality failed;
            # per-modality health still comes from the CSVs, which is why we don't gate on this alone.
            "subprocess_ok": fcomp["returncode"] == 0,
            "wall_s": round(fcomp["wall_s"], 0), "val_exists": val.get("exists"),
            "n_nan_auc_of_grid": val.get("n_nan_auc"), "n_grid_points": val.get("n_grid_points"),
            "max_error_rate": val.get("max_error_rate"), "any_fold_error_flag": val.get("any_fold_error_flag"),
            "best_alpha": val.get("best_alpha"), "alpha_at_boundary": val.get("alpha_at_grid_boundary"),
            "test_all_nan": test.get("all_metrics_nan"), "risk_score_nan_frac": risk_nan,
            "stdout_tags": tags_seen(fcomp["stdout"]),
        })

summary_df = pl.DataFrame(summary_rows)
summary_df

## 5. Pre-fit rejections (skip report)

`_write_skip_report` appends to `SURV_PATH/results/skipped_events/*.jsonl` and is never
truncated -- only rows written **during this run** are shown (tracked by line-count offset
captured before the run above).

In [ ]:
new_rows = new_skip_report_rows()
if new_rows:
    print(f"{len(new_rows)} new skip-report entries this run:")
    for row in new_rows:
        print(f"  {row}")
else:
    print("No pre-fit rejections written this run.")

## 6. Verdict

A row is flagged if: the subprocess failed outright, `*_val.csv`/`*_test.csv` is missing,
every grid point has a NaN AUC(t), the final test metrics are all-NaN, or the risk-score NaN
fraction looks like a silently-failed fold (>5% is used as a conservative trigger -- a single
failed fold out of 5 is ~20%, so this catches partial failures too, not just total ones).

In [ ]:
def row_is_flagged(row: dict) -> bool:
    if not row["subprocess_ok"]:
        return True
    if not row["val_exists"]:
        return True
    if row["n_grid_points"] and row["n_nan_auc_of_grid"] == row["n_grid_points"]:
        return True
    if row["test_all_nan"]:
        return True
    if row["risk_score_nan_frac"] is not None and row["risk_score_nan_frac"] > 0.05:
        return True
    return False


summary_df = summary_df.with_columns(pl.Series("flagged", [row_is_flagged(row) for row in summary_df.iter_rows(named=True)]))
issues_df = summary_df.filter(pl.col("flagged"))

n_skip_rejections = len(new_rows)
verdict_pass = issues_df.is_empty() and n_skip_rejections == 0

print("=" * 72)
print("PASS" if verdict_pass else "FAIL", "-- smoke test verdict")
print("=" * 72)
print(f"{len(summary_df)} rows checked, {len(issues_df)} flagged, {n_skip_rejections} pre-fit rejections.")

if not issues_df.is_empty():
    print("\nFlagged rows:")
    display_cols = ["category", "scheme", "event", "run_type", "model_or_modality",
                     "subprocess_ok", "val_exists", "n_nan_auc_of_grid", "test_all_nan",
                     "risk_score_nan_frac"]
    print(issues_df.select(display_cols))

if not verdict_pass:
    print("\nDo not submit the real SLURM arrays until these are understood -- a green sbatch")
    print("run does not imply healthy fits (see the notebook intro for why).")